In [1]:
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
import tensorflow as tf
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from torchvision import datasets, transforms
import matplotlib.pyplot as plt

def load_and_clean(filename):
    return np.loadtxt(filename, skiprows=2)

data_x = load_and_clean('x24x24.txt')
data_y = load_and_clean('y24x24.txt')
data_z = load_and_clean('z24x24.txt')
full_data = np.vstack([data_x, data_y, data_z])

X = full_data[:, :576]
y = full_data[:, 578]

label_encoder = LabelEncoder()
y = label_encoder.fit_transform(y)

X_train, X_val, y_train, y_val = train_test_split(
    X, y,
    test_size=0.20,
    random_state=67,
    stratify=y           # photos of a person will be in both sets
)

In [2]:
from scipy.ndimage import rotate, shift
import numpy as np
import pandas as pd

def augment_image_flat(x):
    img = x.reshape(24, 24)

    # mały obrót
    angle = np.random.uniform(-5, 5)
    img = rotate(img, angle, reshape=False, mode="nearest")

    # małe przesunięcie max 1-2 piksele
    dx = np.random.uniform(-1, 1)
    dy = np.random.uniform(-1, 1)
    img = shift(img, shift=(dy, dx), mode="nearest")

    return img.reshape(-1)


def augument_df(X_train, y_train):
    X_aug = [X_train]
    y_aug = [y_train]


    unique_classes, counts = np.unique(y_train, return_counts=True)
    target = 100
    for y, count in zip(unique_classes,counts):
       X = X_train[y_train==y]
       to_fill = target-count
       new_img = []
       for c in range(to_fill):
            img = X[np.random.randint(len(X))]
            new_img.append(augment_image_flat(img))
       if len(new_img)>0:
            X_aug.append(np.array(new_img))
            y_aug.append(np.full(to_fill, y))

    return np.vstack(X_aug), np.concatenate(y_aug)


In [3]:
batch_size = 64
learning_rate = 0.001
epochs = 100

X_train, y_train = augument_df(X_train, y_train)

X_train_cnn = X_train.reshape(-1, 1, 24, 24)
X_val_cnn = X_val.reshape(-1, 1, 24, 24)

X_train_tensor = torch.tensor(X_train_cnn, dtype=torch.float32)
X_val_tensor = torch.tensor(X_val_cnn, dtype=torch.float32)

y_train_tensor = torch.tensor(y_train, dtype=torch.long)
y_val_tensor = torch.tensor(y_val, dtype=torch.long)

train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
test_dataset = TensorDataset(X_val_tensor, y_val_tensor)

train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = torch.utils.data.DataLoader(test_dataset, batch_size=batch_size, shuffle=False)


In [4]:
class CNN(nn.Module):
    def __init__(self, hidden_size=128, num_classes=48):
        super().__init__()

        self.conv_layers = nn.Sequential(
            nn.Conv2d(1, 16, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(16, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Dropout2d(0.2)
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(32 * 6 * 6, hidden_size),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(hidden_size, num_classes)

        )

    def forward(self, x):
        x = self.conv_layers(x)
        x = self.classifier(x)
        return x

In [5]:
num_classes = len(np.unique(y))

model = CNN(
    hidden_size=128,
    num_classes=num_classes
)

In [6]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

In [7]:
train_losses = []
val_losses = []

train_accuracies = []
val_accuracies = []

for epoch in range(epochs):
    #training
    model.train()

    running_train_loss = 0.0
    correct_train = 0
    total_train = 0

    for X_batch, y_batch in train_loader:
        optimizer.zero_grad()

        outputs = model(X_batch)
        loss = criterion(outputs, y_batch)

        loss.backward()
        optimizer.step()

        running_train_loss += loss.item() * X_batch.size(0)

        predictions = torch.argmax(outputs, dim=1)
        correct_train += (predictions == y_batch).sum().item()
        total_train += y_batch.size(0)

    epoch_train_loss = running_train_loss / total_train
    epoch_train_accuracy = correct_train / total_train

    train_losses.append(epoch_train_loss)
    train_accuracies.append(epoch_train_accuracy)

   # validation
    model.eval()

    running_val_loss = 0.0
    correct_val = 0
    total_val = 0

    with torch.no_grad():
        for X_batch, y_batch in val_loader:
            outputs = model(X_batch)
            loss = criterion(outputs, y_batch)

            running_val_loss += loss.item() * X_batch.size(0)

            predictions = torch.argmax(outputs, dim=1)
            correct_val += (predictions == y_batch).sum().item()
            total_val += y_batch.size(0)

    epoch_val_loss = running_val_loss / total_val
    epoch_val_accuracy = correct_val / total_val

    val_losses.append(epoch_val_loss)
    val_accuracies.append(epoch_val_accuracy)

    if (epoch + 1) % 10 == 0:
        print(
            f"Epoch {epoch + 1}/{epochs} | "
            f"train loss: {epoch_train_loss:.4f} | "
            f"val loss: {epoch_val_loss:.4f} | "
            f"train acc: {epoch_train_accuracy:.4f} | "
            f"val acc: {epoch_val_accuracy:.4f}"
        )

Epoch 10/100 | train loss: 1.2334 | val loss: 0.9895 | train acc: 0.6354 | val acc: 0.7125
Epoch 20/100 | train loss: 0.8187 | val loss: 0.6794 | train acc: 0.7414 | val acc: 0.8098
Epoch 30/100 | train loss: 0.6313 | val loss: 0.5735 | train acc: 0.7967 | val acc: 0.8317
Epoch 40/100 | train loss: 0.4754 | val loss: 0.5463 | train acc: 0.8458 | val acc: 0.8486
Epoch 50/100 | train loss: 0.4274 | val loss: 0.4980 | train acc: 0.8553 | val acc: 0.8530
Epoch 60/100 | train loss: 0.3762 | val loss: 0.5039 | train acc: 0.8721 | val acc: 0.8661
Epoch 70/100 | train loss: 0.3378 | val loss: 0.5247 | train acc: 0.8847 | val acc: 0.8625
Epoch 80/100 | train loss: 0.3078 | val loss: 0.5026 | train acc: 0.8961 | val acc: 0.8698
Epoch 90/100 | train loss: 0.2840 | val loss: 0.5106 | train acc: 0.9037 | val acc: 0.8734
Epoch 100/100 | train loss: 0.2376 | val loss: 0.5370 | train acc: 0.9160 | val acc: 0.8727


In [8]:
# for epoch in range(epochs):
#     model.train()
#     running_loss = 0.0
#
#     for X_batch, y_batch in train_loader:
#         optimizer.zero_grad()
#
#         outputs = model(X_batch)
#         loss = criterion(outputs, y_batch)
#
#         loss.backward()
#         optimizer.step()
#
#         running_loss += loss.item()
#
#     avg_loss = running_loss / len(train_loader)
#
#     if (epoch + 1) % 10 == 0:
#         print(f"Epoch {epoch + 1}/{epochs}, loss: {avg_loss:.4f}")

In [9]:
model.eval()

correct = 0
total = 0

with torch.no_grad():
    for X_batch, y_batch in val_loader:
        outputs = model(X_batch)
        predictions = torch.argmax(outputs, dim=1)

        correct += (predictions == y_batch).sum().item()
        total += y_batch.size(0)

accuracy = correct / total

print(f"Validation accuracy: {accuracy * 100:.2f}%")

Validation accuracy: 87.27%
